# Module 3, Activity 3: Ingesting Other Data Formats

Until this point we have strictly been working with text (or CSV that we read in like text).  However, as you begin creating RAG applications you will obviously want to consider many other data formats.  This notebook will quickly walk you through a few common ones for inclusion in your vector store.

In [ ]:
!pip install pypdf
!pip install openpyxl

In [ ]:
import boto3, json, time
import io
import tempfile
import pandas as pd
from openpyxl import load_workbook
from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth
from langchain.text_splitter import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain.vectorstores import OpenSearchVectorSearch
from langchain.chains import RetrievalQA
from langchain_community.document_loaders import PyPDFLoader, PyPDFDirectoryLoader
from langchain_aws import ChatBedrock, ChatBedrockConverse
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_aws import BedrockEmbeddings
from langchain.docstore.document import Document

## Some helper functions

We have been getting raw text from S3 with the `get_data_from_s3` helper function.  Now we will add a few more helper functions for dealing with PDFs and Excel files.  These functions will get the files from S3 and then convert them into a list where each entry is in the LangChain Document format, the required input for creating embeddings.

In [ ]:
def get_data_from_s3(bucket_name, key):
    s3 = boto3.client(
        's3',
        region_name=region,
    )
    response = s3.get_object(Bucket=bucket_name, Key=key)
    data = response['Body'].read().decode('utf-8')

    return data

In [ ]:
def get_pdf_docs_from_s3(bucket_name, key):

    s3 = boto3.client('s3', region_name=region)
    response = s3.get_object(Bucket=bucket_name, Key=key)
    pdf_bytes = response['Body'].read()
    
    # Write PDF bytes into a temporary file
    with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as tmp_file:
        tmp_file.write(pdf_bytes)
        tmp_file.flush()  # Ensure data is written to disk
        
        # Pass the path of the temporary file to PyPDFLoader
        loader = PyPDFLoader(tmp_file.name)
        documents = loader.load()
    
    return documents

In [ ]:
def get_excel_from_s3(bucket_name, key):

    s3 = boto3.client("s3", region_name=region)
    response = s3.get_object(Bucket=bucket_name, Key=key)
    excel_io = io.BytesIO(response["Body"].read())

    # Step 2: Read workbook using openpyxl
    wb = load_workbook(excel_io, data_only=True)
    documents = []

    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        headers = [cell.value for cell in next(ws.iter_rows(min_row=1, max_row=1))]
        
        for row in ws.iter_rows(min_row=2, values_only=True):
            row_dict = dict(zip(headers, row))
            content = "\n".join(f"{k}: {v}" for k, v in row_dict.items() if v is not None)
            metadata = {**row_dict, "sheet_name": sheet_name}
            documents.append(Document(page_content=content, metadata=metadata))

    return documents

In [ ]:
session = boto3.session.Session()
region = session.region_name
bedrock_runtime = boto3.client("bedrock-runtime", region_name=region)

In [ ]:
documents = get_pdf_docs_from_s3("bucket-test-cj", "BILL-Q2-25-Press-Release-2-6-25.pdf")
documents[0:3]

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 100
)

chunks = text_splitter.split_documents(documents)
len(chunks)

In [ ]:
documents = get_excel_from_s3("bucket-test-cj", "sample_excel_data.xlsx")

In [ ]:
excel_chunks = text_splitter.split_documents(documents)
len(excel_chunks)

## Concluding thoughts

It is worth experimenting with the splitting, especially when it comes to creating embeddings around tabular data.  Recalling that LangChain will preferentially split on `["\n\n", "\n", ".", " ", ""]`, you might find that your tables are being split in some unusual places.